In [1]:
cd /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane

[Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane'
/content


In [2]:
!pip -q install ultralytics opencv-python pillow matplotlib tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.7 MB/s eta 0:00:00


In [3]:
import torch
import torchvision.transforms as T

from ultralytics import YOLO
from ultralytics.data.dataset import ClassificationDataset
from ultralytics.models.yolo.classify import ClassificationTrainer, ClassificationValidator


class CustomizedDataset(ClassificationDataset):
    """A customized dataset class for image classification with enhanced data augmentation transforms."""

    def __init__(self, root: str, args, augment: bool = False, prefix: str = ""):
        """Initialize a customized classification dataset with enhanced data augmentation transforms."""
        super().__init__(root, args, augment, prefix)

        # Add your custom training transforms here
        train_transforms = T.Compose(
            [
                T.Resize((args.imgsz, args.imgsz)),
                T.RandomHorizontalFlip(p=args.fliplr),
                T.RandomVerticalFlip(p=args.flipud),
                T.RandAugment(interpolation=T.InterpolationMode.BILINEAR),
                T.ColorJitter(brightness=args.hsv_v, contrast=args.hsv_v, saturation=args.hsv_s, hue=args.hsv_h),
                T.ToTensor(),
                T.Normalize(mean=torch.tensor(0), std=torch.tensor(1)),
                T.RandomErasing(p=args.erasing, inplace=True),
            ]
        )

        # Add your custom validation transforms here
        val_transforms = T.Compose(
            [
                T.Resize((args.imgsz, args.imgsz)),
                T.ToTensor(),
                T.Normalize(mean=torch.tensor(0), std=torch.tensor(1)),
            ]
        )
        self.torch_transforms = train_transforms if augment else val_transforms


class CustomizedTrainer(ClassificationTrainer):
    """A customized trainer class for YOLO classification models with enhanced dataset handling."""

    def build_dataset(self, img_path: str, mode: str = "train", batch=None):
        """Build a customized dataset for classification training and the validation during training."""
        return CustomizedDataset(root=img_path, args=self.args, augment=mode == "train", prefix=mode)


class CustomizedValidator(ClassificationValidator):
    """A customized validator class for YOLO classification models with enhanced dataset handling."""

    def build_dataset(self, img_path: str, mode: str = "train"):
        """Build a customized dataset for classification standalone validation."""
        return CustomizedDataset(root=img_path, args=self.args, augment=mode == "train", prefix=self.args.split)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# โหลดโมเดล
model = YOLO("yolo11n-cls.pt") # แนะนำให้ใช้ yolov8n-cls.pt ที่เป็นเวอร์ชันล่าสุด

# Train โมเดลโดยใช้ไฟล์ config.yaml
model.train(data="Faulty_solar_panel_split",  # <--- แก้ตรงนี้!
            trainer=CustomizedTrainer,
            epochs=10,
            imgsz=224,
            batch=64)

Ultralytics 8.3.214 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Faulty_solar_panel_split, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7defb3443dd0>
curves: []
curves_results: []
fitness: 0.9310344755649567
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8620689511299133, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9310344755649567}
save_dir: PosixPath('/content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/runs/classify/train4')
speed: {'preprocess': 0.0731448965508343, 'inference': 0.36445154022783693, 'loss': 7.02183904482892e-05, 'postprocess': 0.00011233333188802239}
task: 'classify'
top1: 0.8620689511299133
top5: 1.0

In [ ]:
# โหลดโมเดล
model2 = YOLO("yolo11n-cls.pt") # แนะนำให้ใช้ yolov8n-cls.pt ที่เป็นเวอร์ชันล่าสุด

# Train โมเดลโดยใช้ไฟล์ config.yaml
model2.train(data="Faulty_solar_panel_split",  # <--- แก้ตรงนี้!
            trainer=CustomizedTrainer,
            epochs=10,
            imgsz=224,
            batch=16,
            optimizer="Adam",
            lr0=1e-3,)

Ultralytics 8.3.214 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Faulty_solar_panel_split, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train5, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7defb364d610>
curves: []
curves_results: []
fitness: 0.9367816150188446
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8735632300376892, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9367816150188446}
save_dir: PosixPath('/content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/runs/classify/train5')
speed: {'preprocess': 0.1695988965512624, 'inference': 0.7336942873588742, 'loss': 0.0003107471331594721, 'postprocess': 0.000499045973424211}
task: 'classify'
top1: 0.8735632300376892
top5: 1.0

In [ ]:
model3 = YOLO("yolo11n-cls.pt") # แนะนำให้ใช้ yolov8n-cls.pt ที่เป็นเวอร์ชันล่าสุด

# Train โมเดลโดยใช้ไฟล์ config.yaml
model3.train(data="Faulty_solar_panel_split",  # <--- แก้ตรงนี้!
            trainer=CustomizedTrainer,
            epochs=15,
            imgsz=224,
            batch=8,
            optimizer="Adam",
            lr0=1e-3,)

Ultralytics 8.3.214 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Faulty_solar_panel_split, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train9, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7def776014f0>
curves: []
curves_results: []
fitness: 0.9425287246704102
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8850574493408203, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9425287246704102}
save_dir: PosixPath('/content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/runs/classify/train9')
speed: {'preprocess': 0.2826641379236197, 'inference': 2.612332540237908, 'loss': 0.0006605402119732035, 'postprocess': 0.002743356329945569}
task: 'classify'
top1: 0.8850574493408203
top5: 1.0

In [ ]:
model3.val(
    data="Faulty_solar_panel_split",
    validator=CustomizedValidator,
    imgsz=224,
    batch=8,
    split="val",   # ค่าเริ่มต้นคือ "val"
)

Ultralytics 8.3.214 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
train: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/train... found 749 images in 5 classes ✅ 
val: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/val... found 87 images in 5 classes ✅ 
test: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/test... found 49 images in 5 classes ✅ 
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 106.6±121.5 MB/s, size: 448.5 KB)
val: Scanning /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/val... 87 images, 0 corrupt: 100% ━━━━━━━━━━━━ 87/87 143.3Kit/s 0.0s
val: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/val/Dusty/Dust (17).jpg: corrupt JPEG restored and saved
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 11/11 2.4it/s 4.6s
                   all      0.874        

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7def778db710>
curves: []
curves_results: []
fitness: 0.9367816150188446
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8735632300376892, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9367816150188446}
save_dir: PosixPath('/content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/runs/classify/val3')
speed: {'preprocess': 0.11002725288521126, 'inference': 4.684567551730547, 'loss': 0.07357745976307718, 'postprocess': 0.0022141494318721117}
task: 'classify'
top1: 0.8735632300376892
top5: 1.0

In [ ]:
model2.val(
    data="Faulty_solar_panel_split",
    validator=CustomizedValidator,
    imgsz=224,
    batch=8,
    split="val",   # ค่าเริ่มต้นคือ "val"
)

Ultralytics 8.3.214 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n-cls summary (fused): 47 layers, 1,532,429 parameters, 0 gradients, 3.2 GFLOPs
train: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/train... found 749 images in 5 classes ✅ 
val: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/val... found 87 images in 5 classes ✅ 
test: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/test... found 49 images in 5 classes ✅ 
val: Fast image access ✅ (ping: 0.5±0.2 ms, read: 119.2±146.0 MB/s, size: 448.5 KB)
val: Scanning /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/val... 87 images, 0 corrupt: 100% ━━━━━━━━━━━━ 87/87 190.0Kit/s 0.0s
val: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/val/Dusty/Dust (17).jpg: corrupt JPEG restored and saved
               classes   top1_acc   to

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7df07c1960c0>
curves: []
curves_results: []
fitness: 0.9080459773540497
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8160919547080994, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9080459773540497}
save_dir: PosixPath('/content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/runs/classify/val4')
speed: {'preprocess': 0.09489259770199931, 'inference': 2.145306632182733, 'loss': 0.0013580115121620674, 'postprocess': 0.002141195422489378}
task: 'classify'
top1: 0.8160919547080994
top5: 1.0

In [ ]:
model.val(
    data="Faulty_solar_panel_split",
    validator=CustomizedValidator,
    imgsz=224,
    batch=8,
    split="val",   # ค่าเริ่มต้นคือ "val"
)

Ultralytics 8.3.214 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n-cls summary (fused): 47 layers, 1,532,429 parameters, 0 gradients, 3.2 GFLOPs
train: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/train... found 749 images in 5 classes ✅ 
val: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/val... found 87 images in 5 classes ✅ 
test: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/test... found 49 images in 5 classes ✅ 
val: Fast image access ✅ (ping: 0.5±0.2 ms, read: 110.3±136.6 MB/s, size: 448.5 KB)
val: Scanning /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/val... 87 images, 0 corrupt: 100% ━━━━━━━━━━━━ 87/87 181.5Kit/s 0.0s
val: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/Faulty_solar_panel_split/val/Dusty/Dust (17).jpg: corrupt JPEG restored and saved
               classes   top1_acc   to

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7def77a6ddc0>
curves: []
curves_results: []
fitness: 0.9080459773540497
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8160919547080994, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9080459773540497}
save_dir: PosixPath('/content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/runs/classify/val5')
speed: {'preprocess': 0.0913568965293, 'inference': 1.933793206868965, 'loss': 0.0013864712531065524, 'postprocess': 0.0024727930928026484}
task: 'classify'
top1: 0.8160919547080994
top5: 1.0

In [ ]:
import os
from pathlib import Path
import cv2
import numpy as np


# ================== CONFIG ==================
MODEL_PATH = "runs/classify/train9/weights/best.pt"  #
TEST_DIR   = Path("Faulty_solar_panel_split/test")
OUT_DIR    = Path("runs/cls_test_vis")   # โฟลเดอร์เซฟภาพพร้อม overlay
TOPK       = 5          # แสดง top-k
SAVE_OUT   = True       # True = บันทึกไฟล์ overlay
SHOW_N     = 16         # แสดงรูปกี่รูปในโน้ตบุ๊ก (ตั้ง None เพื่อไม่แสดง)
N_COLS     = 4          # จำนวนคอลัมน์เวลาวางภาพในกริด
IMG_SIZE   = 224        # ขนาดอินพุตให้โมเดล
# ============================================

def get_image_paths(root: Path):
    exts = (".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff")
    return sorted([p for p in root.rglob("*") if p.suffix.lower() in exts])

def idx2name_map(names):
    # รองรับทั้ง list/dict ตามเวอร์ชันของ ultralytics
    if isinstance(names, dict):
        return names
    return {i: n for i, n in enumerate(names)}

def draw_text_lines(img_bgr, lines, x=10, y=10, font=cv2.FONT_HERSHEY_SIMPLEX,
                    scale=0.6, color=(255,255,255), thick=2, bg=(0,0,0), alpha=0.5, line_gap=24):
    """วาดหลายบรรทัดพร้อมพื้นหลังโปร่งใสให้อ่านง่าย"""
    for line in lines:
        (w, h), base = cv2.getTextSize(line, font, scale, thick)
        overlay = img_bgr.copy()
        cv2.rectangle(overlay, (x-3, y-3), (x + w + 6, y + h + 6), bg, -1)
        cv2.addWeighted(overlay, alpha, img_bgr, 1 - alpha, 0, img_bgr)
        cv2.putText(img_bgr, line, (x, y + h), font, scale, color, thick, cv2.LINE_AA)
        y += line_gap
    return img_bgr

def predict_one_image(model, img_bgr, topk=5, imgsz=224):
    """รับภาพ BGR -> ทำนายคลาส -> คืน (indices, confs) ยาวไม่เกิน topk"""
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    results = model.predict(rgb, imgsz=imgsz, verbose=False)
    res = results[0]
    probs = getattr(res, "probs", None)
    if probs is None or probs.data is None:
        return None, None
    # ถ้ามี top5/top5conf ก็ใช้เลย ไม่งั้นคัดเอง
    if hasattr(probs, "top5") and hasattr(probs, "top5conf"):
        idxs = probs.top5[:topk]
        confs = probs.top5conf[:topk]
    else:
        p = probs.data.detach().cpu().numpy().flatten()
        idxs = p.argsort()[-topk:][::-1]
        confs = p[idxs]
    return idxs, confs


print("[INFO] Loading model...")
model_test = YOLO(MODEL_PATH)
class_names = idx2name_map(model.names)

test_images = get_image_paths(TEST_DIR)
print(f"[INFO] Found {len(test_images)} images in test/")
if len(test_images) == 0:
    raise SystemExit("No test images found. Check TEST_DIR.")

[INFO] Loading model...
[INFO] Found 49 images in test/


In [ ]:
from tqdm import tqdm
vis_paths = []  # เก็บพาธไฟล์ที่บันทึกแล้ว เผื่อใช้ต่อ
for img_path in tqdm(test_images, desc="Predict & save overlays"):
    bgr = cv2.imread(str(img_path))
    if bgr is None:
        print(f"[WARN] Cannot read: {img_path}")
        continue

    idxs, confs = predict_one_image(model, bgr, topk=TOPK, imgsz=IMG_SIZE)
    if idxs is None:
        print(f"[WARN] No probabilities for: {img_path}")
        continue

    # เตรียมข้อความ
    lines = []
    for rank, (idx, cf) in enumerate(zip(idxs, confs), start=1):
        cls_name = class_names.get(int(idx), str(idx))
        lines.append(f"{rank}. {cls_name}: {float(cf)*100:.2f}%")

    # วาด overlay
    bgr_out = bgr.copy()
    bgr_out = draw_text_lines(bgr_out, lines, x=10, y=10, scale=0.6, thick=2,
                              color=(255,255,255), bg=(0,0,0), alpha=0.5, line_gap=24)

    if SAVE_OUT:
        rel = img_path.relative_to(TEST_DIR)
        save_path = OUT_DIR / rel
        save_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(save_path), bgr_out)
        vis_paths.append(save_path)

print(f"[DONE] Saved {len(vis_paths)} overlays to: {OUT_DIR.resolve()}" if SAVE_OUT else "[DONE] Done.")


Predict & save overlays: 100%|██████████| 49/49 [00:24<00:00,  2.02it/s]

[DONE] Saved 49 overlays to: /content/drive/MyDrive/Colab Notebooks/Yolo-cls-solar-plane/runs/cls_test_vis


In [ ]:
import matplotlib.pyplot as plt
# แสดงตัวอย่างภาพ overlay ในกริด (อ่านจากไฟล์ที่เพิ่งบันทึก)
if SHOW_N:
    to_show = vis_paths[:SHOW_N] if SAVE_OUT else test_images[:SHOW_N]
    n = len(to_show)
    n_cols = N_COLS
    n_rows = int(np.ceil(n / n_cols))
    plt.figure(figsize=(4*n_cols, 4*n_rows))
    for i, p in enumerate(to_show, 1):
        img_bgr = cv2.imread(str(p)) if SAVE_OUT else cv2.imread(str(p))
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        ax = plt.subplot(n_rows, n_cols, i)
        ax.imshow(img_rgb)
        ax.set_title(str(Path(p).name), fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

Output hidden; open in https://colab.research.google.com to view.